In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

## Hyperparameters

In [ ]:
##Hyperparams
batch_size = 64 #How many independent sequences will process in parallel?
block_size = 256 #How much should be token length?
max_iter = 15000
eval_intervals = 500
ler_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
eval_iter = 200
n_emb = 48
n_head = 6
n_layer = 6
dropout = 0.2

In [ ]:
torch.manual_seed(1337)

## Load Input Text

In [ ]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [ ]:
text

## Vocabulary

In [ ]:
chars = sorted(list(set(text)))
dic_len = len(chars)
print("vocabulary: ", ''.join(chars))
print("dic_len:", dic_len)

## Encode-Decode Functions

In [ ]:
stoi = {st:i for i,st in enumerate(chars)}
itos = {i:st for i,st in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l : ''.join([itos[i] for i in l])

## Train-Test Split

In [ ]:
data = torch.tensor(encode(text), dtype = torch.long)
n = int(0.9*len(data))
train = data[:n]
test = data[n:]

In [ ]:
print(data[:1000])

In [ ]:
data = torch.tensor(encode(text), dtype= torch.long)

In [ ]:
n = int(0.9 * len(data))
train_data = data[:n]
test_data = data[n:]

## Get batches

In [ ]:
def get_batches(split):
  data1 = train_data if split == 'train' else test_data
  ix = torch.randint(len(data1) - block_size, (batch_size,))
  x = torch.stack([data1[i:i+block_size] for i in ix])
  y = torch.stack([data1[i+1:i+block_size+1] for i in ix])
  x, y = x.to(device), y.to(device)
  return x, y

## Loss Estimation

In [ ]:
@torch.no_grad()
def estimate_loss():
  out = {}
  model.eval()
  for split in ['train', 'test']:
    losses = torch.zeros(eval_iter)
    for k in range(eval_iter):
      X,Y = get_batches(split)
      logits, loss = model(X,Y)
      losses[k] = loss.item()
    out[split] = losses.mean().item()
  model.train()
  return out

## Single Head Attention

In [ ]:
class head(nn.Module):
  def __init__(self, head_size):
    super().__init__()
    self.key = nn.Linear(n_emb, head_size, bias = False)
    self.query = nn.Linear(n_emb, head_size, bias = False)
    self.value = nn.Linear(n_emb, head_size, bias = False)
    self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    #input = B x T X C
    #output = B X T X HS (Head Size)
    B,T,C = x.shape #batch x time state x channel
    k = self.key(x) # B T HS
    q = self.query(x) # B T HS
    wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 #(B x T x Hs @ B x Hs x T --> B x T x T )/ 1/sqrt(dim(k))
    wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) #Mask for masked self attention
    wei = F.softmax(wei, dim=-1) #Softmax Stage
    wei = self.dropout(wei)
    v = self.value(x)
    out = wei @ v #Self Attention Output (Msked weights @ value vectors)
    return out

## Multihead Attention

In [ ]:
class MultiHeadAttention(nn.Module):

  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([head(head_size) for _ in range(num_heads)])
    self.proj = nn.Linear(head_size * num_heads, n_emb)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    out = torch.cat([h(x) for h in self.heads], dim = -1)
    out = self.dropout(self.proj(out))
    return out

## Feedforward Layer

In [ ]:
class Feedforward(nn.Module):

  def __init__(self, n_emb):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_emb, 4 * n_emb),
        nn.ReLU(),
        nn.Linear(4 * n_emb, n_emb),
        nn.Dropout(dropout)
    )

  def forward(self, x):
    return self.net(x)

## Multihead Attention Blocks

In [ ]:
class Block(nn.Module):
  def __init__(self, n_emb, n_head):
    super().__init__()
    head_size = n_emb // n_head
    self.sa = MultiHeadAttention(n_head, head_size)
    self.ff = Feedforward(n_emb)
    self.ln1 = nn.LayerNorm(n_emb)
    self.ln2 = nn.LayerNorm(n_emb)

  def forward(self, x):
    x = x + self.sa(self.ln1(x))
    x = x + self.ff(self.ln2(x))
    return x

## GPT Model

In [ ]:
class GPTmodel(nn.Module):

  def __init__(self):
    super().__init__()
    self.token_emb_table = nn.Embedding(dic_len, n_emb)
    self.position_emb_table = nn.Embedding(block_size, n_emb)
    self.blocks = nn.Sequential(*[Block(n_emb, n_head = n_head) for _ in range(n_layer)])
    self.ln_final = nn.LayerNorm(n_emb)
    self.lm_head = nn.Linear(n_emb, dic_len)
    self.apply(self._init_weights)

  def _init_weights(self, module):
    if isinstance(module, nn.Linear):
      torch.nn.init.normal_(module.weight, mean = 0.0, std = 0.02)
      if module.bias is not None:
        torch.nn.init.zeros_(module.bias)
    if isinstance(module, nn.Embedding):
      torch.nn.init.normal_(module.weight, mean = 0.0, std = 0.02)

  def forward(self, idx, targets = None):
    B, T = idx.shape
    tok_emb = self.token_emb_table(idx)
    tok_pos = self.position_emb_table(torch.arange(T, device = device))
    x = tok_emb + tok_pos
    x = self.blocks(x)
    x = self.ln_final(x)
    logits = self.lm_head(x)

    if targets is None:
      loss = None
      return logits, loss
    else:
      B, T, C = logits.shape
      logits = logits.view(B*T, C)
      targets = targets.view(B * T)
      loss = F.cross_entropy(logits, targets)
      return logits, loss

  def generate(self, idx, max_new_tokens):
    for _ in range(max_new_tokens):
      idx_cond = idx[:, -block_size:]
      logits, loss = self(idx_cond)
      logits = logits[:, -1, :]
      prob = F.softmax(logits, dim = -1)
      idx_next = torch.multinomial(prob, num_samples=1)
      idx = torch.cat((idx, idx_next), dim = 1)
    return idx

## Training GPT Model

In [ ]:
model = GPTmodel()
m = model.to(device)
print(sum(p.numel() for p in m.parameters())/1e6, 'M Parameters')

optimizer = torch.optim.AdamW(model.parameters(), lr = ler_rate)

for iter in range(max_iter):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_intervals == 0 or iter == max_iter - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['test']:.4f}")


    xb,yb = get_batches('train')

    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none = True)
    loss.backward()
    optimizer.step()

## Text Generation

In [ ]:
context = torch.zeros((1,1), dtype = torch.long, device = device)
print(decode(m.generate(context, max_new_tokens = 1000)[0].tolist()))
#open('generated.txt', 'w').write(decode(m.generate(context, max_new_tokens=10000)[0].tolist()))